# KNN algorithm

## Tải thư viện

In [ ]:
! pip install scikit-learn
! pip install numpy
! pip install opencv-python mediapipe
! pip install "protobuf<4"

: 

## Chuẩn bị dữ liệu

### Dataset

Lấy từ bộ data sau: https://www.kaggle.com/datasets/kapitanov/hagrid

### Đọc file json

In [1]:
import numpy as np

def clean_axis(raw_landmarks):
    data = np.array(raw_landmarks)

    x_root = data[0][0]
    y_root = data[0][1]

    data[:, 0] -= x_root
    data[:, 1] -= y_root

    return data.flatten().tolist()

In [2]:
import json 
import os

NUM_CHANGE = {
    'call': 0,
    'dislike': 1,
    'fist': 2,
    'four': 3,
    'like': 4,
    'mute': 5,
    'ok': 6
}

def get_data(JSON_FOLDER):
    X_temp = []
    y_temp = []

    for file_name in os.listdir(JSON_FOLDER):
        class_name = file_name.replace('.json', '')

        if class_name not in NUM_CHANGE:
            continue
        
        path = os.path.join(JSON_FOLDER, file_name)
        with open(path, 'r') as file:
            data = json.load(file)

        for id_photo, infomation in data.items():
            labels = infomation['labels']
            landmarks = infomation['landmarks']

            for i, cur_labels in enumerate(labels):
                if labels[i] == class_name:
                    if i < len(landmarks):
                        raw = landmarks[i]

                        try:
                            clean = clean_axis(raw)
                            X_temp.append(clean)
                            y_temp.append(NUM_CHANGE[class_name])
                        except:
                            continue
    return X_temp, y_temp

### Set up dữ liệu train

In [3]:
JSON_FOLDER_TRAIN = 'Dataset\\ann_train_val'
JSON_FOLDER_TEST = 'Dataset\\ann_test'

X_train, y_train = get_data(JSON_FOLDER_TRAIN)
X_test, y_test = get_data(JSON_FOLDER_TEST)

### Bắt đầu train

In [4]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# fit
knn = KNeighborsClassifier(n_neighbors=11)
knn.fit(X_train, y_train)

# Dự đoán thử
y_pre = knn.predict(X_test)

# Kiểm tra thử độ chính xác mô hình
accuracy = accuracy_score(y_test, y_pre)
n_train = len(X_train)
n_test = len(X_test)
print(f"Độ chính xác của dữ liệu là {accuracy}")
print(f"Số mẫu dữ liệu train {n_train}")
print(f"Số mẫu dữ liệu test {n_test}")

Độ chính xác của dữ liệu là 0.9945907808090311
Số mẫu dữ liệu train 195686
Số mẫu dữ liệu test 17008


## Đóng gói mô hình

In [5]:
import pickle

file_model_name = "knn_model.pkl"

with open(file_model_name, 'wb') as file:
    pickle.dump(knn, file)

# Mediapipe nhận diện xương tay (giúp lấy dữ liệu từ cam và test mô hình trực quan hơn)

In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import pickle
import time
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

MODEL_KNN_PATH = 'knn_model.pkl'      
TASK_FILE_PATH = 'hand_landmarker.task' 

# Map nhãn (Khớp với lúc train)
LABEL_MAP = {
    0: 'Call',
    1: 'Dislike',
    2: 'Fist',
    3: 'Four',
    4: 'Like',
    5: 'Mute',
    6: 'OK'
}

def clean_axis(raw_landmarks):
    """Chuyển đổi tọa độ để KNN hiểu"""
    data = np.array(raw_landmarks)
    base_x, base_y = data[0][0], data[0][1]
    data[:, 0] = data[:, 0] - base_x
    data[:, 1] = data[:, 1] - base_y
    return data.flatten().tolist()

with open(MODEL_KNN_PATH, 'rb') as f:
    knn_model = pickle.load(f)

base_options = python.BaseOptions(model_asset_path=TASK_FILE_PATH)
options = vision.HandLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.VIDEO, 
    num_hands=1,                           
    min_hand_detection_confidence=0.5
)
detector = vision.HandLandmarker.create_from_options(options)

cap = cv2.VideoCapture(0)
cap.set(3, 1280)
cap.set(4, 720)

print("Bấm 'q' để thoát.")

while cap.isOpened():
    success, frame = cap.read()
    if not success: break

    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)

    timestamp_ms = int(time.time() * 1000)
    detection_result = detector.detect_for_video(mp_image, timestamp_ms)

    display_text = "..."
    box_color = (100, 100, 100)

    if detection_result.hand_landmarks:
        for landmarks in detection_result.hand_landmarks:
            h, w, _ = frame.shape
            for lm in landmarks:
                cx, cy = int(lm.x * w), int(lm.y * h)
                cv2.circle(frame, (cx, cy), 5, (0, 255, 0), -1)
            
            try:
                lm_list = []
                for lm in landmarks:
                    lm_list.append([lm.x, lm.y])
                
                cleaned_data = clean_axis(lm_list)

                prediction = knn_model.predict([cleaned_data])
                class_id = prediction[0]
                
                display_text = LABEL_MAP.get(class_id, "Unknown")
                box_color = (0, 200, 255) 
            except Exception as e:
                print(f"Lỗi KNN: {e}")

    cv2.rectangle(frame, (0, 0), (300, 80), box_color, -1)
    cv2.putText(frame, f"Type: {display_text}", (10, 50), 
                cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 255, 255), 3)

    cv2.imshow('Test KNN', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

: 